In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
!pip install -q transformers librosa soundfile accelerate fastapi uvicorn python-multipart pyngrok sentencepiece

In [ ]:
import torch
import transformers
import librosa
import soundfile as sf

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Librosa:", librosa.__version__)
print("CUDA:", torch.cuda.is_available())

PyTorch: 2.14.1+cu130
Transformers: 5.22.1
Librosa: 0.11.0
CUDA: True


In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
print("Whisper model exists:", "model" in globals())
print("Whisper processor exists:", "processor" in globals())
print("FastAPI app exists:", "app" in globals())

GPU available: True
Whisper model exists: False
Whisper processor exists: False
FastAPI app exists: False


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Nepali_Whisper_Project")

MODEL_DIR = PROJECT_DIR / "whisper_small_fft" / "full_training" / "final_model"

print("Model directory:", MODEL_DIR)
print("Directory exists:", MODEL_DIR.exists())

if MODEL_DIR.exists():
    print("\nModel files:")
    for file in MODEL_DIR.iterdir():
        print(file.name)

Mounted at /content/drive
Model directory: /content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_fft/full_training/final_model
Directory exists: True

Model files:
config.json
generation_config.json
model.safetensors
preprocessor_config.json
training_args.bin
tokenizer_config.json
tokenizer.json
processor_config.json


In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_DIR = str(MODEL_DIR)

# Load processor
processor = WhisperProcessor.from_pretrained(MODEL_DIR)

# Load fine-tuned model
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16,
    attn_implementation="sdpa"
)

# Move model to GPU
model = model.to("cuda")
model.eval()

print("Model loaded successfully!")
print("Device:", next(model.parameters()).device)
print("Data type:", next(model.parameters()).dtype)
print("Language:", model.config)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Data type: torch.float16
Language: WhisperConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "apply_spec_augment": false,
  "architectures": [
    "WhisperForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "begin_suppress_tokens": [
    220,
    50257
  ],
  "bos_token_id": 50257,
  "classifier_proj_size": 256,
  "d_model": 768,
  "decoder_attention_heads": 12,
  "decoder_ffn_dim": 3072,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 12,
  "decoder_start_token_id": 50258,
  "dropout": 0.0,
  "dtype": "float16",
  "encoder_attention_heads": 12,
  "encoder_ffn_dim": 3072,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 12,
  "eos_token_id": 50257,
  "forced_decoder_ids": [
    [
      1,
      50259
    ],
    [
      2,
      50359
    ],
    [
      3,
      50363
    ]
  ],
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "mask_feature_length": 10,
  "mask_feature_min_masks": 0,
  "mask_feature_prob": 0.0,
 

In [ ]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

TRANSLATION_MODEL_NAME = "facebook/nllb-200-distilled-600M"

translation_tokenizer = AutoTokenizer.from_pretrained(
    TRANSLATION_MODEL_NAME
)

translation_tokenizer.src_lang = "npi_Deva"

translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATION_MODEL_NAME,
    torch_dtype=torch.float16
).to("cuda")

translation_model.eval()

ENGLISH_TOKEN_ID = translation_tokenizer.convert_tokens_to_ids(
    "eng_Latn"
)

print("NLLB loaded successfully!")
print("Device:", next(translation_model.parameters()).device)
print("Source language:", translation_tokenizer.src_lang)
print("Target token ID:", ENGLISH_TOKEN_ID)


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (5 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

NLLB loaded successfully!
Device: cuda:0
Source language: npi_Deva
Target token ID: 256047


In [ ]:
import librosa
import torch
import time

def transcribe_audio(audio_path):

    start_time = time.time()

    # 1. Load audio at 16 kHz, mono
    audio, sr = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    # 2. Extract Whisper features
    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        device="cuda",
        dtype=torch.float16
    )

    # 3. Generate transcription
    with torch.inference_mode():
        predicted_ids = model.generate(
            input_features,
            language="ne",
            task="transcribe",
            use_cache=True
        )

    # 4. Decode predictions
    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    processing_time = time.time() - start_time

    return {
        "transcription": transcription,
        "processing_time": round(processing_time, 3),
        "audio_duration": round(len(audio) / sr, 3)
    }


print("Transcription function ready!")

Transcription function ready!


In [ ]:

def translate_text(nepali_text):
    if not nepali_text or not nepali_text.strip():
        return ""

    inputs = translation_tokenizer(
        nepali_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.inference_mode():
        translated_tokens = translation_model.generate(
            **inputs,
            forced_bos_token_id=ENGLISH_TOKEN_ID,
            max_new_tokens=256,
            max_length=None,
            num_beams=4
        )

    english_text = translation_tokenizer.batch_decode(
        translated_tokens,
        skip_special_tokens=True
    )[0]

    return english_text


In [ ]:
test_text = "मलाई नेपाली खाना धेरै मन पर्छ।"

english = translate_text(test_text)

print("Nepali:", test_text)
print("English:", english)

Nepali: मलाई नेपाली खाना धेरै मन पर्छ।
English: I love Nepali food very much.


In [ ]:

def process_audio(audio_path):
    # Step 1: Nepali transcription
    whisper_result = transcribe_audio(audio_path)

    nepali_text = whisper_result["transcription"]

    # Step 2: English translation
    english_text = translate_text(nepali_text)

    # Step 3: Return combined results
    return {
        "nepali_transcription": nepali_text,
        "english_translation": english_text,
        "audio_duration": whisper_result["audio_duration"],
        "processing_time": whisper_result["processing_time"]
    }


In [ ]:
from google.colab import files

uploaded = files.upload()

audio_path = next(iter(uploaded.keys()))

print("Uploaded file:", audio_path)

Saving Recording (1).m4a to Recording (1).m4a
Uploaded file: Recording (1).m4a


In [ ]:
result = transcribe_audio(audio_path)

print("===== TRANSCRIPTION RESULT =====")
print("Transcript:", result["transcription"])
print("Audio duration:", result["audio_duration"], "seconds")
print("Processing time:", result["processing_time"], "seconds")

/tmp/ipykernel_1413/539001295.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


===== TRANSCRIPTION RESULT =====
Transcript: क्यो कामपुरा गर्न एक महिनाको अवधि लाग्छ
Audio duration: 4.587 seconds
Processing time: 23.0 seconds


In [ ]:
# Warm-up run
_ = transcribe_audio(audio_path)

# Benchmark run
torch.cuda.synchronize()
start_time = time.time()

result = transcribe_audio(audio_path)

torch.cuda.synchronize()
benchmark_time = time.time() - start_time

print("===== SECOND INFERENCE =====")
print("Transcript:", result["transcription"])
print("Audio duration:", result["audio_duration"], "seconds")
print("Processing time:", round(benchmark_time, 2), "seconds")
print("RTF:", round(
    benchmark_time / result["audio_duration"], 3
))

/tmp/ipykernel_1413/539001295.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1413/539001295.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


===== SECOND INFERENCE =====
Transcript: क्यो कामपुरा गर्न एक महिनाको अवधि लाग्छ
Audio duration: 4.587 seconds
Processing time: 0.7 seconds
RTF: 0.152


In [ ]:
!pip install -q fastapi uvicorn python-multipart pyngrok

In [ ]:

from fastapi import FastAPI, UploadFile, File, HTTPException
import tempfile
import os
import time

app = FastAPI(title="NepTranscript API")

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model": "Whisper Small FFT",
        "translation_model": "NLLB-200"
    }

@app.get("/")
def home():
    return {
        "status": "running",
        "model": "Whisper Small FFT",
        "translation_model": "NLLB-200",
        "features": ["Nepali transcription", "English translation"]
    }


@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):

    if not file.filename:
        raise HTTPException(
            status_code=400,
            detail="Please upload a file."
        )

    allowed_extensions = {
        ".wav", ".mp3", ".m4a", ".ogg", ".flac"
    }

    extension = os.path.splitext(file.filename)[1].lower()

    if extension not in allowed_extensions:
        raise HTTPException(
            status_code=400,
            detail="Unsupported audio format."
        )

    temp_path = None

    try:
        suffix = os.path.splitext(file.filename)[1] or ".audio"

        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=suffix
        ) as temp:
            temp.write(await file.read())
            temp_path = temp.name

        start_time = time.time()

        result = process_audio(temp_path)

        total_processing_time = round(
            time.time() - start_time, 3
        )

        return {
            "filename": file.filename,
            "nepali_transcription": result["nepali_transcription"],
            "english_translation": result["english_translation"],
            "audio_duration": result["audio_duration"],
            "processing_time": total_processing_time
        }

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)


In [ ]:
from getpass import getpass
from pyngrok import ngrok

# Enter the token when prompted; it will not be displayed or stored in the notebook.
NGROK_AUTH_TOKEN = getpass("Enter your ngrok authtoken: ")

if not NGROK_AUTH_TOKEN.strip():
    raise ValueError("An ngrok authentication token is required.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("ngrok authentication configured successfully!")


In [ ]:
import threading
import uvicorn

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print("FastAPI server started!")

FastAPI server started!


In [ ]:
import requests

try:
    response = requests.get(
        "http://127.0.0.1:8000/",
        timeout=5
    )
    print("FastAPI server is running!")
    print(response.json())

except requests.exceptions.RequestException:
    print("FastAPI server is not responding.")

INFO:     127.0.0.1:46868 - "GET / HTTP/1.1" 200 OK
FastAPI server is running!
{'status': 'running', 'model': 'Whisper Small FFT', 'translation_model': 'NLLB-200', 'features': ['Nepali transcription', 'English translation']}


### Create a public ngrok tunnel

The generated URL is publicly reachable while the tunnel is active. Share it only with people who should be able to access your test service.

In [ ]:
from pyngrok import ngrok

# Expose the FastAPI server
public_url = ngrok.connect(8000).public_url

print("Public API URL:", public_url)
print("API documentation:", public_url + "/docs")

## Notes and limitations

- This notebook is a Colab-based GPU deployment workflow. The local NepTranscript application uses its own backend and local model paths.
- The API accepts audio files with the extensions validated in the endpoint code.
- The translation output depends on NLLB-200 and may contain lexical or contextual errors; review the English translation when accuracy is important.
- The public ngrok URL is temporary and should not be treated as a production deployment.
- Do not upload private audio, dataset files, model checkpoints, API credentials, or generated prediction files to the public repository.

